# Brain-AI Smoke Test & Development Validation

**Objective**: Fast but meaningful smoke test for the `brain_automl` package forecasting pipeline.

| Item | Detail |
|---|---|
| **Dataset** | `examples/dataset.csv` — multi-stock daily OHLCV with pre-computed features |
| **Scope** | Last 1 year (~252 trading days) per stock |
| **Targets** | `Close` (price) and `Volatility` (squared log returns) |
| **Horizon** | 30 trading days |
| **AutoGluon** | `best_quality` preset (Chronos-2, DeepAR, PatchTST, ETS, ARIMA, Theta, ensemble) |
| **StatsForecast** | AutoARIMA, AutoETS, AutoCES, AutoTheta, SeasonalNaive, SeasonalWindowAverage |
| **FLAML** | Tree-based AutoML (LightGBM/XGBoost via ts_forecast_regression) |
| **Foundation Models** | MOIRAI-2, TimesFM (standalone if installed) |
| **Backtesting** | Expanding window (3 non-overlapping windows) |
| **Metrics** | RMSE, MAE, MAPE, MASE, CRPS, Directional Accuracy |

Architecture compatibility: follows registry-driven, protocol-based design from
`ARCHITECTURE_MIGRATION_GUIDE.md` and `MULTIMODAL_ARCHITECTURE_PLAN.md`.

In [1]:
# ═══════════════════════════════════════════════════════════════
# Environment Setup, Imports, Reproducibility
# ═══════════════════════════════════════════════════════════════

from pathlib import Path
import json
import logging
import os
import subprocess
import sys
import time
import warnings
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

# ── CPU-only runtime (stable on M4 Mac) ─────────────────────
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "0"
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "4")

# ── Verbose logging for Lightning (if used by optional deps) ──
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logging.getLogger("pytorch_lightning").setLevel(logging.INFO)
logging.getLogger("lightning.pytorch").setLevel(logging.INFO)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

np.random.seed(SEED)

# ── Local source path ────────────────────────────────────────
PROJECT_ROOT = Path.cwd().resolve().parent
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# ── brain_automl imports (registry-driven architecture) ──────
from brain_automl.config import get_default_config
from brain_automl.core import BACKEND_REGISTRY, ModalityResult
from brain_automl.model_zoo.time_series_ai import TimeSeriesAutoML
from brain_automl.model_zoo.time_series_ai.data_preparation import (
    to_standard_timeseries_format,
    to_autogluon_timeseries_format,
    compute_forecast_metrics,
    DEFAULT_BUSINESS_SEASONALITY,
)
from brain_automl.forecasting.foundation.models import list_foundation_models, get_foundation_model_spec
from brain_automl.metrics import directional_accuracy

warnings.filterwarnings("ignore")

# ── Paths & global config ────────────────────────────────────
DATA_PATH = Path.cwd() / "dataset.csv"
OUTPUT_DIR = Path.cwd() / "brain_automl_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Forecasting parameters ───────────────────────────────────
HORIZON = 30               # 30 trading days forecast horizon
LAST_N_DAYS = 252          # ~1 year of business days
DATE_COL = "Date"
TARGET_COL = "Close"
GROUP_COL = "Stock"
VOLATILITY_COL = "Volatility"
TARGETS = [TARGET_COL, VOLATILITY_COL]

# ── AutoGluon configuration ──────────────────────────────────
# Switched from best_quality to high_quality for faster smoke-test iterations.
# Transformer-family models are also excluded later in fit(...).
AG_PRESETS = "high_quality"
AG_TIME_LIMIT = 600           # seconds per predictor
AG_EVAL_METRIC = "MASE"       # primary evaluation metric
MAX_STOCKS_SMOKE_TEST = 1     # limit stocks for speed

print(f"Project root : {PROJECT_ROOT}")
print(f"Data path    : {DATA_PATH} (exists: {DATA_PATH.exists()})")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"Runtime      : CPU only (CUDA/MPS disabled)")
print(f"Random seed  : {SEED}")
print(f"Horizon      : {HORIZON} trading days")
print(f"AG presets   : {AG_PRESETS} (time_limit={AG_TIME_LIMIT}s)")
print("Logger setup : INFO for pytorch_lightning, lightning.pytorch")

Project root : /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai
Data path    : /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/dataset.csv (exists: True)
Output dir   : /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/brain_automl_output
Runtime      : CPU only (CUDA/MPS disabled)
Random seed  : 42
Horizon      : 30 trading days
AG presets   : high_quality (time_limit=600s)
Logger setup : INFO for pytorch_lightning, lightning.pytorch


## §1. Data Loading & Preparation
Load `dataset.csv`, trim to the last ~252 business days per stock, and compute/validate the Volatility column as squared log returns: $\text{vol}_t = \left(\ln \frac{C_t}{C_{t-1}}\right)^2$.

In [2]:
# ═══════════════════════════════════════════════════════════════
# §1. Data Loading & Preparation
# ═══════════════════════════════════════════════════════════════

df_raw = pd.read_csv(DATA_PATH)
df_raw.columns = [c.strip() for c in df_raw.columns]
print(f"Raw dataset: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} cols")
print(f"Columns: {list(df_raw.columns)}")

# Parse dates and sort
df_raw[DATE_COL] = pd.to_datetime(df_raw[DATE_COL], errors="coerce")
df_raw = df_raw.dropna(subset=[DATE_COL]).sort_values([GROUP_COL, DATE_COL]).reset_index(drop=True)

# ── Trim to last ~252 business days per stock ────────────────
df = (
    df_raw
    .groupby(GROUP_COL, group_keys=False)
    .apply(lambda g: g.tail(LAST_N_DAYS))
    .reset_index(drop=True)
)

# ── Compute volatility as squared log returns ────────────────
# vol_t = (log(close_t / close_{t-1}))^2
df["_vol_computed"] = (
    np.log(df[TARGET_COL] / df.groupby(GROUP_COL)[TARGET_COL].shift(1))
) ** 2

# Use existing Volatility column if present; fill gaps with computed
if VOLATILITY_COL in df.columns:
    df[VOLATILITY_COL] = df[VOLATILITY_COL].fillna(df["_vol_computed"])
else:
    df[VOLATILITY_COL] = df["_vol_computed"]

# Drop first row per stock (NaN from shift) and clean up
df = df.dropna(subset=[VOLATILITY_COL]).reset_index(drop=True)
df = df.drop(columns=["_vol_computed"])

# ── Identify stocks ──────────────────────────────────────────
all_stocks = sorted(df[GROUP_COL].unique().tolist())
smoke_test_stocks = all_stocks[:MAX_STOCKS_SMOKE_TEST]

print(f"\nTrimmed dataset: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"Date range: {df[DATE_COL].min().date()} → {df[DATE_COL].max().date()}")
print(f"Unique stocks ({len(all_stocks)}): {all_stocks}")
print(f"Smoke test stocks ({len(smoke_test_stocks)}): {smoke_test_stocks}")
print(f"\nRows per stock:")
print(df.groupby(GROUP_COL).size().to_string())
print(f"\nTargets: {TARGETS}  |  Horizon: {HORIZON}")
display(df.head(3))

Raw dataset: 18,381 rows × 12 cols
Columns: ['Date', 'Close', 'High', 'Low', 'Open', 'Volume', 'Return', 'MA_5', 'MA_10', 'MA_20', 'Volatility', 'Stock']

Trimmed dataset: 1,260 rows × 12 cols
Date range: 2023-12-21 → 2024-12-31
Unique stocks (5): ['BANKNIFTY', 'HDFCBANK', 'NIFTY50', 'RELIANCE', 'TCS']
Smoke test stocks (1): ['BANKNIFTY']

Rows per stock:
Stock
BANKNIFTY    252
HDFCBANK     252
NIFTY50      252
RELIANCE     252
TCS          252

Targets: ['Close', 'Volatility']  |  Horizon: 30


,Date,Close,High,Low,Open,Volume,Return,MA_5,MA_10,MA_20,Volatility,Stock
0,2023-12-21,47840.148438,47932.398438,46919.699219,47172.449219,208000,0.008288,47833.519531,47566.594922,46393.774805,0.000069,BANKNIFTY
1,2023-12-22,47491.851562,48071.398438,47415.851562,47837.750000,165800,-0.007307,47703.179688,47589.580078,46589.492383,0.000053,BANKNIFTY
2,2023-12-26,47724.851562,47838.449219,47411.648438,47576.398438,118700,0.004894,47674.610156,47630.640234,46787.279883,0.000024,BANKNIFTY


## §2. Reusable Helper Functions
Metrics suite (CRPS + full metrics), expanding-window backtest engine, and Plotly visualization helpers. These are designed for direct integration into the `brain_automl` package.

In [3]:
# ═══════════════════════════════════════════════════════════════
# §2a. Metrics Helpers (CRPS + Full Suite)
#   → Candidate for brain_automl/metrics/forecasting.py
# ═══════════════════════════════════════════════════════════════


def compute_crps_from_quantiles(
    y_true: np.ndarray,
    quantile_forecasts: Dict[float, np.ndarray],
    quantile_levels: List[float],
) -> float:
    """Approximate CRPS from quantile predictions using pinball loss.

    CRPS ≈ (2/K) * Σ_k  pinball_loss(α_k, y, q_k)

    Parameters
    ----------
    y_true : array-like – actual observed values.
    quantile_forecasts : dict mapping quantile level α → predicted values.
    quantile_levels : sorted list of α values, e.g. [0.1, 0.2, …, 0.9].

    Returns
    -------
    float – approximate CRPS (lower is better).
    """
    y = np.asarray(y_true, dtype=float)
    K = len(quantile_levels)
    if K < 2:
        return float("nan")
    total = 0.0
    for alpha in quantile_levels:
        q = np.asarray(quantile_forecasts[alpha], dtype=float)
        err = y - q
        total += np.mean(np.where(err >= 0, alpha * err, (alpha - 1.0) * err))
    return float(2.0 * total / K)


def get_quantile_columns(pred_df: pd.DataFrame) -> Tuple[List[str], List[float]]:
    """Extract quantile column names and levels from AutoGluon predictions."""
    q_cols, q_levels = [], []
    for c in pred_df.columns:
        if c == "mean":
            continue
        try:
            q = float(c)
            q_cols.append(c)
            q_levels.append(q)
        except (ValueError, TypeError):
            continue
    pairs = sorted(zip(q_levels, q_cols))
    return [c for _, c in pairs], [q for q, _ in pairs]


def compute_full_metrics(
    y_train: np.ndarray,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    quantile_forecasts: Optional[Dict[float, np.ndarray]] = None,
    quantile_levels: Optional[List[float]] = None,
    seasonality: int = DEFAULT_BUSINESS_SEASONALITY,
) -> Dict[str, float]:
    """Compute RMSE, MAE, MAPE, MASE, CRPS, Directional Accuracy.

    Wraps brain_automl.model_zoo.time_series_ai.data_preparation.compute_forecast_metrics
    and adds CRPS + directional accuracy on top.
    """
    m = compute_forecast_metrics(y_train, y_true, y_pred, seasonality=seasonality)
    m["directional_accuracy"] = directional_accuracy(
        np.asarray(y_true), np.asarray(y_pred)
    )
    if quantile_forecasts and quantile_levels:
        try:
            m["crps"] = compute_crps_from_quantiles(
                np.asarray(y_true), quantile_forecasts, quantile_levels
            )
        except Exception:
            m["crps"] = float("nan")
    return m


print("Metric helpers defined: compute_crps_from_quantiles, get_quantile_columns, compute_full_metrics")

Metric helpers defined: compute_crps_from_quantiles, get_quantile_columns, compute_full_metrics


In [4]:
# ═══════════════════════════════════════════════════════════════
# §2b. Expanding Window Backtest Engine & Visualization Helpers
#   → Candidates for brain_automl/model_zoo/time_series_ai/backtesting.py
#     and brain_automl/utilities/plotting.py
# ═══════════════════════════════════════════════════════════════


def expanding_window_backtest(
    series_data: pd.DataFrame,
    forecast_fn: Callable,
    horizon: int = 30,
    n_windows: int = 3,
    min_train_size: int = 120,
    seasonality: int = DEFAULT_BUSINESS_SEASONALITY,
) -> Dict[str, Any]:
    """Run expanding-window backtesting on one time series.

    Parameters
    ----------
    series_data : DataFrame with [unique_id, ds, y].
    forecast_fn : callable(train_df, horizon) → dict
        Must return {'predictions': array} and optionally
        {'quantile_forecasts': dict, 'quantile_levels': list}.
    horizon, n_windows, min_train_size, seasonality : usual meaning.

    Returns
    -------
    dict  {'per_window': DataFrame, 'aggregate': dict of mean metrics}
    """
    n = len(series_data)
    # Non-overlapping expanding windows anchored at the end
    window_ends = []
    for i in range(n_windows):
        end = n - i * horizon
        if end - horizon >= min_train_size:
            window_ends.append(end)
    window_ends = sorted(window_ends)

    if not window_ends:
        raise ValueError(
            f"Insufficient data for backtest: n={n}, horizon={horizon}, "
            f"min_train={min_train_size}"
        )

    results = []
    for w_idx, end in enumerate(window_ends):
        train = series_data.iloc[: end - horizon].copy().reset_index(drop=True)
        test = series_data.iloc[end - horizon : end].copy().reset_index(drop=True)
        h = len(test)
        if h == 0:
            continue
        try:
            fc = forecast_fn(train, h)
            preds = np.asarray(fc["predictions"][:h], dtype=float)
            m = compute_full_metrics(
                y_train=train["y"].values,
                y_true=test["y"].values,
                y_pred=preds,
                quantile_forecasts=fc.get("quantile_forecasts"),
                quantile_levels=fc.get("quantile_levels"),
                seasonality=seasonality,
            )
            m.update(
                window=w_idx,
                train_size=len(train),
                test_size=h,
                test_start=str(test["ds"].iloc[0].date()),
                test_end=str(test["ds"].iloc[-1].date()),
                status="ok",
            )
            results.append(m)
        except Exception as exc:
            results.append(dict(
                window=w_idx, train_size=len(train), test_size=h,
                test_start=str(test["ds"].iloc[0].date()) if h else "N/A",
                test_end=str(test["ds"].iloc[-1].date()) if h else "N/A",
                status=f"error: {exc}",
            ))

    rdf = pd.DataFrame(results)
    ok = rdf[rdf["status"] == "ok"]
    if not ok.empty:
        skip = {"window", "train_size", "test_size", "test_start", "test_end", "status"}
        agg = ok[[c for c in ok.columns if c not in skip]].mean().to_dict()
        agg["n_windows"] = len(ok)
    else:
        agg = {"n_windows": 0}
    return {"per_window": rdf, "aggregate": agg}


# ── Visualization helpers ────────────────────────────────────
PLOT_COLORS = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
    "#9467bd", "#8c564b", "#e377c2", "#17becf",
]


def plot_forecast_vs_actual(
    dates, actual, forecasts,
    title="Forecast vs Actual",
    train_dates=None, train_values=None,
    quantile_lo=None, quantile_hi=None,
):
    """Plotly chart: actual vs one or more forecast lines, optional CI band."""
    fig = go.Figure()
    if train_dates is not None and train_values is not None:
        ctx = min(60, len(train_dates))
        fig.add_trace(go.Scatter(
            x=train_dates[-ctx:], y=train_values[-ctx:],
            name="Train (context)", line=dict(color="gray", width=1, dash="dot"),
        ))
    fig.add_trace(go.Scatter(
        x=dates, y=actual, name="Actual", line=dict(color="black", width=2),
    ))
    for idx, (name, preds) in enumerate(forecasts.items()):
        fig.add_trace(go.Scatter(
            x=dates, y=preds, name=name,
            line=dict(color=PLOT_COLORS[idx % len(PLOT_COLORS)], width=1.5, dash="dash"),
        ))
    if quantile_lo is not None and quantile_hi is not None:
        fig.add_trace(go.Scatter(x=dates, y=quantile_hi, mode="lines",
                                 line=dict(width=0), showlegend=False))
        fig.add_trace(go.Scatter(x=dates, y=quantile_lo, mode="lines",
                                 line=dict(width=0), fill="tonexty",
                                 fillcolor="rgba(31,119,180,0.15)",
                                 name="10th–90th pctl"))
    fig.update_layout(
        title=dict(text=title, font=dict(size=16)),
        xaxis_title="Date", yaxis_title="Value",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        height=450, hovermode="x unified", template="plotly_white",
    )
    return fig


def plot_metric_bars(metrics_df, metrics_to_plot, group_col="model", title="Comparison"):
    """Horizontal bar chart comparing models across chosen metrics."""
    n = len(metrics_to_plot)
    fig = make_subplots(rows=1, cols=n,
                        subplot_titles=[m.upper() for m in metrics_to_plot])
    for ci, met in enumerate(metrics_to_plot):
        if met not in metrics_df.columns:
            continue
        sub = metrics_df[[group_col, met]].dropna().sort_values(met)
        fig.add_trace(go.Bar(
            y=sub[group_col], x=sub[met], orientation="h",
            marker_color=PLOT_COLORS[: len(sub)],
            text=sub[met].round(4), textposition="outside", showlegend=False,
        ), row=1, col=ci + 1)
    fig.update_layout(title=dict(text=title, font=dict(size=16)),
                      height=max(300, 40 * len(metrics_df)),
                      template="plotly_white")
    return fig


print("Helpers defined: expanding_window_backtest, plot_forecast_vs_actual, plot_metric_bars")

Helpers defined: expanding_window_backtest, plot_forecast_vs_actual, plot_metric_bars


## §3. Backend & Foundation Model Availability
Check which `brain_automl` registry backends and foundation time-series models are importable in this environment.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# §3. Backend & Foundation Model Availability
# ═══════════════════════════════════════════════════════════════

config = get_default_config()
configured_backends = config["backends"]["by_modality"]["time_series"]["default"]

import_map = {
    "autogluon_timeseries": "autogluon.timeseries",
    "statsforecast": "statsforecast",
    "flaml": "flaml",
}

def safe_import_check(import_name: Optional[str], timeout: int = 10) -> bool:
    if not isinstance(import_name, str) or not import_name:
        return False
    print(f"  Importing {import_name} ...")
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_ROOT / "src")
    try:
        subprocess.run(
            [sys.executable, "-c", f"import {import_name}"],
            check=True,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=timeout,
        )
        return True
    except subprocess.TimeoutExpired:
        print(f"    {import_name}: timeout after {timeout}s")
        return False
    except Exception as exc:
        print(f"    {import_name}: failed {type(exc).__name__}: {exc}")
        return False

rows = []
print("=== Checking registered time-series backends ===")
for b in configured_backends:
    # NeuralForecast is intentionally disabled in this notebook profile.
    if b == "neuralforecast":
        print("  Checking backend: neuralforecast... skipped (disabled)")
        rows.append(dict(backend=b, registered=False, backend_available=False, direct_import=False))
        continue

    print(f"  Checking backend: {b}...")
    registered = BACKEND_REGISTRY.has(b)
    backend_ok = False
    if registered:
        backend_ok = bool(BACKEND_REGISTRY.get(b).is_available())
    import_name = import_map.get(b) or b
    direct_ok = safe_import_check(import_name, timeout=10)

    rows.append(dict(backend=b, registered=registered,
                     backend_available=backend_ok, direct_import=direct_ok))
    print(f"    registered={registered}, backend_available={backend_ok}, direct_import={direct_ok}")

avail_df = pd.DataFrame(rows)
print("=== brain_automl Registered Backends (time_series) ===")
display(avail_df)

# ── Foundation model catalog ─────────────────────────────────
print("\n=== Foundation Model Catalog ===")
print("Checking foundation model availability one-by-one...")
foundations = ["chronos", "timesfm", "lag_llama", "moirai"]
fm_rows = []
for model_name in foundations:
    spec = get_foundation_model_spec(model_name)
    if spec is None:
        fm_rows.append({"model": model_name, "provider": None, "available": False})
        print(f"  {model_name}: spec not found")
        continue
    import_name = spec.import_path
    print(f"  {model_name}: checking import for {import_name}...")
    t0 = time.time()
    available = safe_import_check(import_name, timeout=10)
    elapsed = time.time() - t0
    fm_rows.append({"model": model_name, "provider": spec.provider, "available": available})
    print(f"    {model_name}: available={available} (took {elapsed:.2f}s)")

fm_df = pd.DataFrame(fm_rows)
print("=== Foundation model availability summary ===")
display(fm_df)

# ── AutoGluon availability ───────────────────────────────────
print("\n=== Importing AutoGluon ===")
AG_AVAILABLE = False
try:
    from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
    AG_AVAILABLE = True
    print("✓ AutoGluon TimeSeriesPredictor available")
except ImportError as e:
    print(f"✗ AutoGluon not available: {e}")

# ── StatsForecast / FLAML availability ───────────────────────
print("\n=== Importing StatsForecast ===")
SF_AVAILABLE = False
try:
    from statsforecast import StatsForecast as _SF
    from statsforecast.models import AutoARIMA, AutoETS, AutoCES, AutoTheta
    from statsforecast.models import SeasonalNaive, WindowAverage, SeasonalWindowAverage
    SF_AVAILABLE = True
    print("✓ StatsForecast available (AutoARIMA, AutoETS, AutoCES, AutoTheta, etc.)")
except ImportError as e:
    print(f"✗ StatsForecast not available: {e}")

print("\n=== NeuralForecast ===")
NF_AVAILABLE = False
NF_REASON = "disabled by notebook configuration"
print("⚠ NeuralForecast disabled by notebook configuration")

print("\n=== Importing FLAML ===")
FLAML_AVAILABLE = False
try:
    from flaml import AutoML as _FLAML_AutoML
    FLAML_AVAILABLE = True
    print("✓ FLAML AutoML available")
except ImportError as e:
    print(f"✗ FLAML not available: {e}")

# ── Standalone foundation model packages ─────────────────────
FM_MOIRAI, FM_TIMESFM, FM_LAGLLAMA = False, False, False
for pkg, label, attr in [
    ("uni2ts", "MOIRAI-2", "FM_MOIRAI"),
    ("timesfm", "TimesFM", "FM_TIMESFM"),
    ("lag_llama", "Lag-Llama", "FM_LAGLLAMA"),
]:
    try:
        __import__(pkg)
        vars()[attr] = True
        globals()[attr] = True
        print(f"✓ {label} ({pkg}) available")
    except ImportError:
        print(f"✗ {label} ({pkg}) not installed — standalone run will be skipped")

=== Checking registered time-series backends ===
  Checking backend: autogluon_timeseries...
  Importing autogluon.timeseries ...
    registered=True, backend_available=True, direct_import=True
  Checking backend: statsforecast...
  Importing statsforecast ...
    registered=True, backend_available=True, direct_import=True
  Checking backend: flaml...
  Importing flaml ...


## §4. AutoGluon `best_quality` Forecasting (Primary Engine)
Fit one `TimeSeriesPredictor` per target (Close, Volatility) using all smoke-test stocks as a multi-item `TimeSeriesDataFrame`.

The `best_quality` preset automatically trains and ensembles **Chronos-2**, DeepAR, PatchTST, ETS, ARIMA, Theta, and other models. Switch to `"high_quality"` if too slow on M4 Mac Mini.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# §4. AutoGluon best_quality Forecasting
#     Fit one predictor per target (Close, Volatility) across
#     all smoke-test stocks as a multi-item TimeSeriesDataFrame.
#     best_quality automatically includes Chronos-2 + strong ensemble.
# ═══════════════════════════════════════════════════════════════

ag_all_results = {}        # target → per-stock metrics DataFrame
ag_leaderboards = {}       # target → AutoGluon leaderboard
ag_predictions_store = {}  # (target, stock) → dict

if not AG_AVAILABLE:
    print("AutoGluon not available — skipping this section.")
else:
    for target in TARGETS:
        print(f"\n{'=' * 70}")
        print(f"TARGET: {target}  |  Stocks: {smoke_test_stocks}  |  Horizon: {HORIZON}")
        print(f"{'=' * 70}")
        print("[Step 1/5] Building target-specific multi-item frame...")

        # ── Build multi-item DataFrame for this target ────────
        tdf = df[df[GROUP_COL].isin(smoke_test_stocks)][[DATE_COL, GROUP_COL, target]].copy()
        tdf = tdf.rename(columns={DATE_COL: "timestamp", GROUP_COL: "item_id", target: "target"})
        tdf["timestamp"] = pd.to_datetime(tdf["timestamp"])
        tdf = tdf.dropna(subset=["target"]).sort_values(["item_id", "timestamp"]).reset_index(drop=True)

        print(f"  Prepared rows for target={target}: {len(tdf):,}")
        print("[Step 2/5] Creating per-stock train/test splits...")

        # ── Train / test split: hold out last HORIZON per item
        train_parts, test_parts = [], []
        for stk in smoke_test_stocks:
            sd = tdf[tdf["item_id"] == stk].copy()
            total_rows = len(sd)
            if total_rows <= HORIZON:
                print(f"  ⚠ {stk}: only {total_rows} rows — skipping (needs > {HORIZON})")
                continue

            train_chunk = sd.iloc[:-HORIZON]
            test_chunk = sd.iloc[-HORIZON:]
            train_parts.append(train_chunk)
            test_parts.append(test_chunk)

            print(
                f"  {stk}: total={total_rows}, train={len(train_chunk)}, test={len(test_chunk)} "
                f"(test range: {test_chunk['timestamp'].iloc[0].date()} → {test_chunk['timestamp'].iloc[-1].date()})"
            )

        if not train_parts or not test_parts:
            print(f"  No valid train/test splits for target={target}. Skipping target.")
            ag_all_results[target] = pd.DataFrame()
            ag_leaderboards[target] = pd.DataFrame()
            continue

        train_pdf = pd.concat(train_parts, ignore_index=True)
        test_pdf = pd.concat(test_parts, ignore_index=True)
        full_pdf = pd.concat([train_pdf, test_pdf], ignore_index=True)
        full_pdf = full_pdf.sort_values(["item_id", "timestamp"]).reset_index(drop=True)

        train_tsdf = TimeSeriesDataFrame.from_data_frame(
            train_pdf, id_column="item_id", timestamp_column="timestamp")
        full_tsdf = TimeSeriesDataFrame.from_data_frame(
            full_pdf, id_column="item_id", timestamp_column="timestamp")

        print(f"  Train rows (all stocks): {len(train_tsdf):,}")
        print(f"  Full rows  (all stocks): {len(full_tsdf):,}")

        # ── Fit predictor ─────────────────────────────────────
        print("[Step 3/5] Training AutoGluon predictor...")
        ag_path = str(OUTPUT_DIR / f"autogluon_{target.lower()}")
        predictor = TimeSeriesPredictor(
            prediction_length=HORIZON,
            target="target",
            eval_metric=AG_EVAL_METRIC,
            freq="B",
            path=ag_path,
            verbosity=4,
        )

        # Ensure the item timestamps are treated as business-day frequency.
        train_tsdf = train_tsdf.convert_frequency(freq="B")
        full_tsdf = full_tsdf.convert_frequency(freq="B")

        excluded_model_types = [
            "Chronos", "ChronosModel", "Chronos2",
            "TemporalFusionTransformer", "PatchTST", "TiDE", "Transformer",
            "DeepAR",
        ]
        t0 = time.time()
        print(f"  Excluding model types from AutoGluon fit: {excluded_model_types}")
        predictor.fit(
            train_data=train_tsdf,
            presets=AG_PRESETS,
            excluded_model_types=excluded_model_types,
            time_limit=AG_TIME_LIMIT,
            random_seed=SEED,
        )
        fit_elapsed = time.time() - t0
        print(f"  Fit completed in {fit_elapsed:.1f}s")

        try:
            model_names = predictor.model_names()
        except Exception:
            model_names = []

        if model_names:
            print(f"  Trained models ({len(model_names)}):")
            for i, model_name in enumerate(model_names, start=1):
                print(f"    {i:>2}. {model_name}")

            blocked_keywords = ["chronos", "transformer", "patchtst", "tide", "tft", "deepar"]
            blocked_models = [
                m for m in model_names if any(k in str(m).lower() for k in blocked_keywords)
            ]
            if blocked_models:
                print(f"  ⚠ Blocked model families still present: {blocked_models}")
            else:
                print("  ✓ No blocked model families detected in trained model list.")
        else:
            print("  Trained models: unavailable from predictor.model_names()")

        # ── Predict ───────────────────────────────────────────
        print("[Step 4/5] Generating ensemble predictions for test windows...")
        predictions = predictor.predict(train_tsdf)

        # ── Leaderboard ───────────────────────────────────────
        try:
            lb = predictor.leaderboard(full_tsdf, silent=True)
            ag_leaderboards[target] = lb
            print(f"\n  Leaderboard ({target} — top 10):")
            display(lb.head(10))

            print("  Individual model validation summary:")
            for _, row in lb.head(20).iterrows():
                model_name = row.get("model", "N/A")
                score_val = row.get("score_val", np.nan)
                pred_time = row.get("pred_time_val", np.nan)
                fit_time = row.get("fit_time_marginal", row.get("fit_time", np.nan))
                print(
                    f"    model={model_name} | score_val={score_val} | "
                    f"fit_time={fit_time} | pred_time={pred_time}"
                )

            lb_blocked = lb[
                lb["model"].astype(str).str.contains(
                    "chronos|transformer|patchtst|tide|tft|deepar", case=False, regex=True, na=False
                )
            ]
            if not lb_blocked.empty:
                print("  ⚠ Blocked-family rows found in leaderboard:")
                display(lb_blocked)
            else:
                print("  ✓ No blocked-family rows found in leaderboard.")
        except Exception as exc:
            print(f"  Leaderboard error: {exc}")
            ag_leaderboards[target] = pd.DataFrame()

        # ── Per-stock metrics + quantile extraction ───────────
        print("[Step 5/5] Testing predictions per stock and computing metrics...")
        q_cols, q_levels = get_quantile_columns(predictions.reset_index())
        print(f"  Quantile columns detected: {q_cols}")

        target_metrics = []
        for stk in smoke_test_stocks:
            try:
                stk_pred = predictions.loc[stk]
                stk_test = test_pdf[test_pdf["item_id"] == stk]
                stk_train = train_pdf[train_pdf["item_id"] == stk]
                if stk_test.empty:
                    print(f"  {stk}: no test rows found after split — skipping")
                    continue

                y_true = stk_test["target"].values
                y_pred = stk_pred["mean"].values[: len(y_true)]
                y_train = stk_train["target"].values

                print(
                    f"  Testing stock={stk}: train_points={len(y_train)}, "
                    f"test_points={len(y_true)}, pred_points={len(y_pred)}"
                )

                # Quantile forecasts for CRPS
                qf = {}
                for ql in q_levels:
                    cn = str(ql)
                    if cn in stk_pred.columns:
                        qf[ql] = stk_pred[cn].values[: len(y_true)]

                m = compute_full_metrics(
                    y_train=y_train, y_true=y_true, y_pred=y_pred,
                    quantile_forecasts=qf or None,
                    quantile_levels=q_levels if qf else None,
                )
                m["stock"] = stk
                m["target"] = target
                m["model"] = "AutoGluon_best_quality"
                target_metrics.append(m)

                # Store for plotting later
                q_lo = qf.get(0.1)
                q_hi = qf.get(0.9)
                ag_predictions_store[(target, stk)] = dict(
                    dates=stk_test["timestamp"].values,
                    actual=y_true, predicted=y_pred,
                    train_dates=stk_train["timestamp"].values,
                    train_values=y_train,
                    quantile_lo=q_lo, quantile_hi=q_hi,
                )

                print(f"  {stk}: RMSE={m['rmse']:.4f}  MAE={m['mae']:.4f}  "
                      f"MASE={m.get('mase', float('nan')):.4f}  "
                      f"DA={m.get('directional_accuracy', float('nan')):.3f}  "
                      f"CRPS={m.get('crps', float('nan')):.4f}")
            except Exception as exc:
                print(f"  {stk}: metrics extraction failed — {exc}")

        ag_all_results[target] = pd.DataFrame(target_metrics)

    # ── Summary ───────────────────────────────────────────────
    print(f"\n{'=' * 70}")
    print("AutoGluon Forecasting — Summary")
    for tgt, rdf in ag_all_results.items():
        if not rdf.empty:
            print(f"\n{tgt}:")
            display(rdf.round(4))


TARGET: Close  |  Stocks: ['BANKNIFTY']  |  Horizon: 30
[Step 1/5] Building target-specific multi-item frame...
  Prepared rows for target=Close: 252
[Step 2/5] Creating per-stock train/test splits...
  BANKNIFTY: total=252, train=222, test=30 (test range: 2024-11-18 → 2024-12-31)
  Train rows (all stocks): 222
  Full rows  (all stocks): 252
[Step 3/5] Training AutoGluon predictor...


Beginning AutoGluon training... Time limit = 600s
AutoGluon will save models to '/Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/brain_automl_output/autogluon_close'
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.13.7
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.4.0: Thu Mar 19 19:31:09 PDT 2026; root:xnu-12377.101.15~1/RELEASE_ARM64_T8132
CPU Count:          10
Pytorch Version:    2.9.1
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.30 GB / 16.00 GB (26.9%)
Disk Space Avail:   768.67 GB / 931.32 GB (82.5%)
Setting presets to: high_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MASE,
 'excluded_model_types': ['Chronos',
                          'ChronosModel',
                          'Chronos2',
  

  Excluding model types from AutoGluon fit: ['Chronos', 'ChronosModel', 'Chronos2', 'TemporalFusionTransformer', 'PatchTST', 'TiDE', 'Transformer', 'DeepAR']


		-1.7885      = Validation score (-MASE)
		0.002   s    = Training runtime
		0.860   s    = Prediction runtime
	-1.7885       = Validation score (-MASE)
	0.01    s     = Training runtime
	0.86    s     = Validation (prediction) runtime
Training timeseries model RecursiveTabular. Training for up to 119.8s of the 599.0s of remaining time.
	Window 0
Shortening all series to at most 1000035
train_df shape: (158, 35), val_df shape: (30, 35)


[50]	valid_set's l1: 0.571523
[100]	valid_set's l1: 0.60605
[150]	valid_set's l1: 0.620514
[200]	valid_set's l1: 0.641081
[250]	valid_set's l1: 0.656879


...

,model,score_test,score_val,pred_time_test,pred_time_val,fit_time_marginal,fit_order
0,DynamicOptimizedTheta,-0.410399,-0.427036,0.015726,0.015105,0.004798,4
1,WeightedEnsemble,-0.417789,-0.412996,0.032813,0.032842,0.050584,6
2,DirectTabular,-0.504317,-0.474430,0.016691,0.017275,0.409978,3
3,RecursiveTabular,-0.511101,-0.639081,0.035997,0.034712,0.413678,2
4,SeasonalNaive,-0.528989,-0.786842,0.016385,0.014805,0.005615,1
5,AutoETS,-3.138989,-5.041282,1.252471,0.063892,0.006199,5


  Individual model validation summary:
    model=DynamicOptimizedTheta | score_val=-0.42703612812715563 | fit_time=0.004797935485839844 | pred_time=0.015105009078979492
    model=WeightedEnsemble | score_val=-0.41299640286265055 | fit_time=0.05058416599058546 | pred_time=0.03284155047731474
    model=DirectTabular | score_val=-0.4744296132476452 | fit_time=0.40997767448425293 | pred_time=0.017275333404541016
    model=RecursiveTabular | score_val=-0.6390811402875521 | fit_time=0.4136776924133301 | pred_time=0.03471207618713379
    model=SeasonalNaive | score_val=-0.7868420496698895 | fit_time=0.005615234375 | pred_time=0.014804840087890625
    model=AutoETS | score_val=-5.041281896026891 | fit_time=0.0061986446380615234 | pred_time=0.06389236450195312
  ✓ No blocked-family rows found in leaderboard.
[Step 5/5] Testing predictions per stock and computing metrics...
  Quantile columns detected: ['0.1', '0.2', '0.3', '0.4', '0.5', '0.6', '0.7', '0.8', '0.9']
  Testing stock=BANKNIFTY: tra

,mae,mse,rmse,mape,smape,wape,mase,r2,directional_accuracy,crps,stock,target,model
0,1683.7069,3.895627e+06,1973.7344,3.1918,3.2627,3.2311,1.9512,-2.3698,0.5172,1379.3408,BANKNIFTY,Close,AutoGluon_best_quality



Volatility:


,mae,mse,rmse,mape,smape,wape,mase,r2,directional_accuracy,crps,stock,target,model
0,0.0001,0.0,0.0001,186370.0343,126.4189,142.9715,0.481,-0.3278,0.3793,0.0001,BANKNIFTY,Volatility,AutoGluon_best_quality


## §5b. StatsForecast + FLAML — Additional Models

Models **not already in AutoGluon's ensemble**:

| Library | Models | Reason to include separately |
|---|---|---|
| **StatsForecast** | AutoETS, AutoCES, AutoTheta, SeasonalWindowAverage | AG uses its own ETS/Theta; SF versions have different implementations |
| **FLAML** | AutoML (ts_forecast_regression) | Tree-based ensemble approach (LightGBM/XGBoost) — complementary to statistical methods |

All run CPU-only.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# §5b. StatsForecast + FLAML
#      Additional models NOT covered by AutoGluon best_quality.
#      All run on first smoke-test stock, Close target, same train/test split.
# ═══════════════════════════════════════════════════════════════

extra_results = []  # list of metric dicts
extra_predictions_store = {}  # model_name → dict(dates, actual, predicted, ...)

# Make this cell runnable standalone even if helper cells were not executed.
if "compute_full_metrics" in globals():
    metrics_fn = compute_full_metrics
else:
    print("⚠ compute_full_metrics not found; using local fallback metrics (no CRPS).")

    def metrics_fn(y_train, y_true, y_pred):
        m = compute_forecast_metrics(y_train, y_true, y_pred, seasonality=DEFAULT_BUSINESS_SEASONALITY)
        m["directional_accuracy"] = directional_accuracy(np.asarray(y_true), np.asarray(y_pred))
        return m

# ── Prepare common train/test for first stock ─────────────────
ext_stock = smoke_test_stocks[0]
ext_df = (
    df[df[GROUP_COL] == ext_stock][[DATE_COL, TARGET_COL]]
    .sort_values(DATE_COL).reset_index(drop=True)
)
ext_df["ds"] = pd.to_datetime(ext_df[DATE_COL])
ext_df["y"] = ext_df[TARGET_COL].astype(float)
ext_df["unique_id"] = ext_stock

# FLAML requires regular frequency. Reindex to business days and fill gaps.
ext_full_idx = pd.date_range(ext_df["ds"].min(), ext_df["ds"].max(), freq="B")
ext_regular = (
    ext_df.set_index("ds")
    .reindex(ext_full_idx)
    .rename_axis("ds")
    .reset_index()
)
ext_regular["unique_id"] = ext_stock
ext_regular["y"] = ext_regular["y"].interpolate(method="linear").ffill().bfill()

ext_train = ext_regular.iloc[:-HORIZON][["unique_id", "ds", "y"]].reset_index(drop=True)
ext_test = ext_regular.iloc[-HORIZON:][["unique_id", "ds", "y"]].reset_index(drop=True)

print(f"Extra backends benchmark stock: {ext_stock}")
print(f"  Train: {len(ext_train)} rows  |  Test: {len(ext_test)} rows")
print("  Note: series reindexed to regular business-day frequency for FLAML compatibility.\n")


# ═══════════════════════════════════════════════════════════════
# A) StatsForecast — AutoETS, AutoCES, AutoTheta, SeasonalWindowAverage
# ═══════════════════════════════════════════════════════════════
if SF_AVAILABLE:
    print("─── StatsForecast ───────────────────────────────────")
    try:
        from statsforecast import StatsForecast as SForecast
        from statsforecast.models import (
            AutoARIMA, AutoETS, AutoCES, AutoTheta,
            SeasonalNaive, Naive, WindowAverage, SeasonalWindowAverage,
        )

        SEASONALITY = 5  # business-week seasonality

        sf_models_list = [
            AutoARIMA(season_length=SEASONALITY),
            AutoETS(season_length=SEASONALITY),
            AutoCES(season_length=SEASONALITY),
            AutoTheta(season_length=SEASONALITY),
            SeasonalNaive(season_length=SEASONALITY),
            SeasonalWindowAverage(season_length=SEASONALITY, window_size=4),
            WindowAverage(window_size=10),
        ]
        sf_model_names = [
            "AutoARIMA", "AutoETS", "AutoCES", "AutoTheta",
            "SeasonalNaive", "SeasWinAvg", "WinAvg10",
        ]

        sf = SForecast(models=sf_models_list, freq="B", n_jobs=1, fallback_model=Naive())
        t0 = time.time()
        sf_forecast = sf.forecast(df=ext_train, h=HORIZON)
        sf_elapsed = time.time() - t0
        print(f"  StatsForecast completed in {sf_elapsed:.1f}s")

        # Extract per-model predictions and compute metrics
        for model_name, col_name in zip(sf_model_names, [type(m).__name__ for m in sf_models_list]):
            if col_name not in sf_forecast.columns:
                matching = [c for c in sf_forecast.columns if col_name.lower() in c.lower()]
                if not matching:
                    print(f"  ⚠ {model_name}: column '{col_name}' not found in forecast → skipping")
                    continue
                col_name = matching[0]

            preds = sf_forecast[col_name].values[:HORIZON]
            m = metrics_fn(
                y_train=ext_train["y"].values,
                y_true=ext_test["y"].values,
                y_pred=preds,
            )
            m.update(stock=ext_stock, target="Close", model=f"SF_{model_name}")
            extra_results.append(m)
            extra_predictions_store[f"SF_{model_name}"] = dict(
                dates=ext_test["ds"].values, actual=ext_test["y"].values, predicted=preds,
                train_dates=ext_train["ds"].values, train_values=ext_train["y"].values,
            )
            print(f"  SF_{model_name}: RMSE={m['rmse']:.4f}  MAE={m['mae']:.4f}  "
                  f"MASE={m.get('mase', float('nan')):.4f}  DA={m.get('directional_accuracy', float('nan')):.3f}")

    except Exception as exc:
        print(f"  StatsForecast error: {exc}")
        import traceback; traceback.print_exc()
else:
    print("StatsForecast: skipped (not installed)")


# ═══════════════════════════════════════════════════════════════
# B) FLAML — Tree-based AutoML (LightGBM/XGBoost ensemble)
# ═══════════════════════════════════════════════════════════════
if FLAML_AVAILABLE:
    print("\n─── FLAML AutoML ───────────────────────────────────")
    try:
        from flaml import AutoML

        flaml_automl = AutoML()
        flaml_train_x = ext_train[["ds"]].copy()
        flaml_train_y = ext_train["y"].copy()
        flaml_test_x = ext_test[["ds"]].copy()

        t0 = time.time()
        flaml_automl.fit(
            X_train=flaml_train_x,
            y_train=flaml_train_y,
            task="ts_forecast_regression",
            period=5,  # business-week seasonality
            time_budget=60,
            metric="mape",
            verbose=0,
        )
        flaml_elapsed = time.time() - t0
        print(f"  FLAML fit completed in {flaml_elapsed:.1f}s")
        print(f"  Best estimator: {flaml_automl.best_estimator}")

        flaml_preds = flaml_automl.predict(flaml_test_x)
        m = metrics_fn(
            y_train=ext_train["y"].values,
            y_true=ext_test["y"].values,
            y_pred=flaml_preds[:HORIZON],
        )
        m.update(stock=ext_stock, target="Close", model=f"FLAML_{flaml_automl.best_estimator}")
        extra_results.append(m)
        extra_predictions_store[f"FLAML_{flaml_automl.best_estimator}"] = dict(
            dates=ext_test["ds"].values, actual=ext_test["y"].values,
            predicted=flaml_preds[:HORIZON],
            train_dates=ext_train["ds"].values, train_values=ext_train["y"].values,
        )
        print(f"  FLAML: RMSE={m['rmse']:.4f}  MAE={m['mae']:.4f}  "
              f"MASE={m.get('mase', float('nan')):.4f}  DA={m.get('directional_accuracy', float('nan')):.3f}")

    except Exception as exc:
        print(f"  FLAML error: {exc}")
        import traceback; traceback.print_exc()
else:
    print("\nFLAML: skipped (not installed)")


# ═══════════════════════════════════════════════════════════════
# Summary
# ═══════════════════════════════════════════════════════════════
extra_results_df = pd.DataFrame(extra_results) if extra_results else pd.DataFrame()
if not extra_results_df.empty:
    print(f"\n{'=' * 70}")
    print(f"Extra Backend Results ({ext_stock} Close) — {len(extra_results)} models")
    print(f"{'=' * 70}")
    disp = [c for c in [
        "model", "rmse", "mae", "mape", "mase", "directional_accuracy", "r2",
    ] if c in extra_results_df.columns]
    display(extra_results_df[disp].sort_values("rmse").round(4))
else:
    print("\nNo extra backend results produced.")

2026-04-05 23:04:28,213 | INFO | flaml.automl.task.time_series_task | Couldn't import Prophet, skipping
2026-04-05 23:04:28,214 | INFO | flaml.automl.task.time_series_task | Couldn't import orbit, skipping


Extra backends benchmark stock: BANKNIFTY
  Train: 239 rows  |  Test: 30 rows
  Note: series reindexed to regular business-day frequency for FLAML compatibility.

─── StatsForecast ───────────────────────────────────
  StatsForecast completed in 0.2s
  SF_AutoARIMA: RMSE=1868.8670  MAE=1568.0593  MASE=1.8734  DA=0.379
  SF_AutoETS: RMSE=1875.6770  MAE=1575.3861  MASE=1.8822  DA=0.000
  ⚠ AutoCES: column 'AutoCES' not found in forecast → skipping
  SF_AutoTheta: RMSE=1722.9828  MAE=1388.9574  MASE=1.6594  DA=0.517
  SF_SeasonalNaive: RMSE=2107.0354  MAE=1826.3910  MASE=2.1821  DA=0.448
  ⚠ SeasWinAvg: column 'SeasonalWindowAverage' not found in forecast → skipping
  SF_WinAvg10: RMSE=1515.4539  MAE=1212.0818  MASE=1.4481  DA=0.000

─── FLAML AutoML ───────────────────────────────────
  FLAML fit completed in 60.1s
  Best estimator: extra_tree
  FLAML: RMSE=1936.6600  MAE=1656.8880  MASE=1.9795  DA=0.483

Extra Backend Results (BANKNIFTY Close) — 6 models


,model,rmse,mae,mape,mase,directional_accuracy,r2
4,SF_WinAvg10,1515.4539,1212.0818,2.2925,1.4481,0.0000,-1.0984
2,SF_AutoTheta,1722.9828,1388.9574,2.6266,1.6594,0.5172,-1.7124
0,SF_AutoARIMA,1868.8670,1568.0593,2.9701,1.8734,0.3793,-2.1912
1,SF_AutoETS,1875.6770,1575.3861,2.9841,1.8822,0.0000,-2.2145
5,FLAML_extra_tree,1936.6600,1656.8880,3.1421,1.9795,0.4828,-2.4269
3,SF_SeasonalNaive,2107.0354,1826.3910,3.4650,2.1821,0.4483,-3.0564


## §7. Results Dashboard
Combined leaderboard, metric comparison bar charts, and forecast-vs-actual plots for 2–3 example stocks.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# §7. Results Dashboard
#     Combined leaderboard, bar-chart comparisons, forecast plots.
# ═══════════════════════════════════════════════════════════════

# Make dashboard cell robust when run independently/out of order.
ag_all_results = globals().get("ag_all_results", {})
fm_results_df = globals().get("fm_results_df", pd.DataFrame())
extra_results_df = globals().get("extra_results_df", pd.DataFrame())
ag_predictions_store = globals().get("ag_predictions_store", {})
extra_predictions_store = globals().get("extra_predictions_store", {})
backtest_results = globals().get("backtest_results", {})
ext_stock = globals().get("ext_stock", smoke_test_stocks[0] if smoke_test_stocks else "N/A")

# ── 1. Combined leaderboard (AG + FM + SF/FLAML) ─────────────
all_metric_frames = []
for tgt, rdf in ag_all_results.items():
    if isinstance(rdf, pd.DataFrame) and not rdf.empty:
        all_metric_frames.append(rdf)
if isinstance(fm_results_df, pd.DataFrame) and not fm_results_df.empty:
    all_metric_frames.append(fm_results_df)
if isinstance(extra_results_df, pd.DataFrame) and not extra_results_df.empty:
    all_metric_frames.append(extra_results_df)

if all_metric_frames:
    combined = pd.concat(all_metric_frames, ignore_index=True)
    disp_cols = [c for c in [
        "stock", "target", "model",
        "rmse", "mae", "mape", "mase", "crps",
        "directional_accuracy", "r2", "smape", "wape",
    ] if c in combined.columns]

    print("=" * 70)
    print("COMBINED FORECASTING LEADERBOARD — ALL ENGINES")
    print("=" * 70)
    display(combined[disp_cols].sort_values("rmse").round(4))

    # ── Bar chart per target ──────────────────────────────────
    for tgt in TARGETS:
        tdata = combined[combined["target"] == tgt].copy()
        if tdata.empty:
            continue
        tdata["label"] = tdata["stock"] + " | " + tdata["model"]
        fig = plot_metric_bars(
            tdata.sort_values("rmse"),
            metrics_to_plot=["rmse", "mae", "mape"],
            group_col="label",
            title=f"Model Comparison — {tgt} (AG + SF + FLAML + FM)",
        )
        fig.show()
else:
    combined = pd.DataFrame()
    print("No metrics collected.")

# ── 2. Forecast vs Actual plots ──────────────────────────────
# AutoGluon predictions (multi-stock, multi-target, with CI bands)
print(f"\n{'=' * 70}")
print("FORECAST PLOTS — AutoGluon")
print("=" * 70)

for stk in smoke_test_stocks[:3]:
    for tgt in TARGETS:
        key = (tgt, stk)
        if key not in ag_predictions_store:
            continue
        s = ag_predictions_store[key]
        fig = plot_forecast_vs_actual(
            dates=pd.to_datetime(s["dates"]),
            actual=s["actual"],
            forecasts={"AutoGluon_best_quality": s["predicted"]},
            title=f"{stk} — {tgt} (horizon={HORIZON})",
            train_dates=pd.to_datetime(s["train_dates"]),
            train_values=s["train_values"],
            quantile_lo=s.get("quantile_lo"),
            quantile_hi=s.get("quantile_hi"),
        )
        fig.show()

# Combined overlay: all extra backend predictions for the benchmark stock
if extra_predictions_store:
    print(f"\n{'=' * 70}")
    print(f"ALL MODELS OVERLAY — {ext_stock} Close")
    print("=" * 70)

    all_forecasts = {}
    # Include AG prediction for the same stock if available
    ag_key = ("Close", ext_stock)
    if ag_key in ag_predictions_store:
        all_forecasts["AutoGluon_best_quality"] = ag_predictions_store[ag_key]["predicted"]
    for model_name, s in extra_predictions_store.items():
        all_forecasts[model_name] = s["predicted"]

    ref = list(extra_predictions_store.values())[0]
    fig = plot_forecast_vs_actual(
        dates=pd.to_datetime(ref["dates"]),
        actual=ref["actual"],
        forecasts=all_forecasts,
        title=f"{ext_stock} — Close — All Models (AG + SF + FLAML)",
        train_dates=pd.to_datetime(ref["train_dates"]),
        train_values=ref["train_values"],
    )
    fig.show()

# ── 3. Backtesting summary ───────────────────────────────────
if backtest_results:
    print(f"\n{'=' * 70}")
    print("EXPANDING WINDOW BACKTEST SUMMARY")
    print("=" * 70)
    bt_rows = []
    for stk, bt in backtest_results.items():
        if isinstance(bt, dict):
            agg = bt.get("aggregate", {}).copy()
            agg["stock"] = stk
            bt_rows.append(agg)
    bt_df = pd.DataFrame(bt_rows)
    if not bt_df.empty:
        bt_disp = [c for c in [
            "stock", "n_windows", "rmse", "mae", "mape", "mase",
            "crps", "directional_accuracy",
        ] if c in bt_df.columns]
        display(bt_df[bt_disp].round(4))

COMBINED FORECASTING LEADERBOARD — ALL ENGINES


,stock,target,model,rmse,mae,mape,mase,directional_accuracy,r2,smape,wape
4,BANKNIFTY,Close,SF_WinAvg10,1515.4539,1212.0818,2.2925,1.4481,0.0000,-1.0984,2.3329,2.3250
2,BANKNIFTY,Close,SF_AutoTheta,1722.9828,1388.9574,2.6266,1.6594,0.5172,-1.7124,2.6803,2.6643
0,BANKNIFTY,Close,SF_AutoARIMA,1868.8670,1568.0593,2.9701,1.8734,0.3793,-2.1912,3.0334,3.0078
1,BANKNIFTY,Close,SF_AutoETS,1875.6770,1575.3861,2.9841,1.8822,0.0000,-2.2145,3.0479,3.0219
5,BANKNIFTY,Close,FLAML_extra_tree,1936.6600,1656.8880,3.1421,1.9795,0.4828,-2.4269,3.2090,3.1782
3,BANKNIFTY,Close,SF_SeasonalNaive,2107.0354,1826.3910,3.4650,2.1821,0.4483,-3.0564,3.5460,3.5034



FORECAST PLOTS — AutoGluon

ALL MODELS OVERLAY — BANKNIFTY Close


In [ ]:
# ═══════════════════════════════════════════════════════════════
# §7b. Save All Results
# ═══════════════════════════════════════════════════════════════

save_dir = OUTPUT_DIR / "smoke_test_results"
save_dir.mkdir(parents=True, exist_ok=True)

# Make save cell robust when run independently/out of order.
combined = globals().get("combined", pd.DataFrame())
extra_results_df = globals().get("extra_results_df", pd.DataFrame())
ag_leaderboards = globals().get("ag_leaderboards", {})
backtest_results = globals().get("backtest_results", {})
fm_results_df = globals().get("fm_results_df", pd.DataFrame())

# Combined leaderboard (all engines)
if not combined.empty:
    p = save_dir / "combined_leaderboard.csv"
    combined.to_csv(p, index=False)
    print(f"Saved leaderboard          : {p}")
else:
    print("Skipped leaderboard save   : `combined` is empty or undefined")

# AutoGluon leaderboards
for tgt, lb in ag_leaderboards.items():
    if lb is not None and not lb.empty:
        p = save_dir / f"ag_leaderboard_{tgt.lower()}.csv"
        lb.to_csv(p, index=False)
        print(f"Saved AG leaderboard ({tgt}) : {p}")

# Extra backend results (SF/FLAML)
if not extra_results_df.empty:
    p = save_dir / "extra_backend_results.csv"
    extra_results_df.to_csv(p, index=False)
    print(f"Saved extra results        : {p}")
else:
    print("Skipped extra results save : `extra_results_df` is empty or undefined")

# Backtest per-window results
for stk, bt in backtest_results.items():
    if isinstance(bt, dict) and "per_window" in bt:
        p = save_dir / f"backtest_{stk.lower()}.csv"
        bt["per_window"].to_csv(p, index=False)
        print(f"Saved backtest ({stk})       : {p}")

# Foundation model results
if not fm_results_df.empty:
    p = save_dir / "foundation_model_results.csv"
    fm_results_df.to_csv(p, index=False)
    print(f"Saved FM results           : {p}")
else:
    print("Skipped FM results save    : `fm_results_df` is empty or undefined")

# Run metadata
meta = dict(
    seed=SEED, horizon=HORIZON, last_n_days=LAST_N_DAYS,
    ag_presets=AG_PRESETS, ag_time_limit=AG_TIME_LIMIT,
    ag_eval_metric=AG_EVAL_METRIC,
    targets=TARGETS, stocks=smoke_test_stocks,
    n_bt_windows=globals().get("N_BT_WINDOWS", None),
    bt_preset=globals().get("BT_PRESET", None),
    ag_available=AG_AVAILABLE,
    sf_available=SF_AVAILABLE, flaml_available=FLAML_AVAILABLE,
    fm_moirai=FM_MOIRAI, fm_timesfm=FM_TIMESFM, fm_lagllama=FM_LAGLLAMA,
    total_models=len(combined) if not combined.empty else 0,
)
(save_dir / "run_metadata.json").write_text(json.dumps(meta, indent=2))
print(f"Saved metadata             : {save_dir / 'run_metadata.json'}")
print(f"\nAll outputs → {save_dir}")

Skipped leaderboard save   : `combined` is empty or undefined
Saved extra results        : /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/brain_automl_output/smoke_test_results/extra_backend_results.csv
Skipped FM results save    : `fm_results_df` is empty or undefined
Saved metadata             : /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/brain_automl_output/smoke_test_results/run_metadata.json

All outputs → /Volumes/MacSSD/01_Projects/Chandravesh-ML-Research/projects/core-research/brain-ai/examples/brain_automl_output/smoke_test_results


## §8. Package Integration Suggestions

### Ready-to-integrate functions from this notebook

| Function | Target location | Purpose |
|---|---|---|
| `compute_crps_from_quantiles()` | `brain_automl/metrics/forecasting.py` | Add CRPS alongside RMSE/MAE/MAPE/MASE |
| `expanding_window_backtest()` | `brain_automl/model_zoo/time_series_ai/backtesting.py` (new) | Generalised backtest engine |
| `plot_forecast_vs_actual()` | `brain_automl/utilities/plotting.py` (new) | Consistent forecast visualisation |
| `plot_metric_bars()` | `brain_automl/utilities/plotting.py` (new) | Side-by-side metric bars |

### Suggested improvements

1. **Add `backtest()` to `TimeSeriesAutoML`** — wrap `expanding_window_backtest` as a first-class method.
2. **Foundation model registry update** — add `moirai-2`, `timesfm-2.5`, `chronos-2` specs with version-aware `is_available()` in `brain_automl/forecasting/foundation/models.py`.
3. **CLI smoke-test command**: `brain-ai smoke-test --data dataset.csv --horizon 30 --presets best_quality`
4. **Test suite** (`tests/test_smoke_forecasting.py`):
   - `test_autogluon_best_quality_smoke()` — quick AG run, `time_limit=30`
   - `test_expanding_window_backtest()` — backtest on synthetic data
   - `test_crps_computation()` — unit test for CRPS calculation
   - `test_multi_target_forecasting()` — Close + Volatility targets
5. **Update `AutoGluonTimeSeriesBackend`** — expose quantile forecasts, AG leaderboard, and configurable presets.